In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

from mlrose_hiive import ContinuousPeaksGenerator
# from mlrose_hiive import SARunner, GARunner, NNGSRunner, RHCRunner, MIMICRunner
from mlrose_hiive import RHCRunner

In [ ]:
EXPERIMENT_NAME = 'RHC_ContinuousPeaks_GridSearch'
GENERATOR = ContinuousPeaksGenerator
SEED = 0
PROBLEM_SIZE_LIST = [8]
ITERATIONS_LIST = [1e9]
MAX_ATTEMPTS_LIST = [50, 100, 500]
RESTART_LIST = [0, 25, 50, 100, 500]
NUM_RUNS = 1

In [ ]:
output_dir = f'metrics/{EXPERIMENT_NAME}'
os.makedirs(output_dir, exist_ok=True)

In [3]:
all_df = pd.DataFrame()
group_i = 0
run_i = 0
for problem_size in PROBLEM_SIZE_LIST:
    for iterations in ITERATIONS_LIST:
        for max_attempts in tqdm(MAX_ATTEMPTS_LIST):
            for restarts in RESTART_LIST:
                for i in range(NUM_RUNS):
                    problem = GENERATOR().generate(seed=SEED+i, size=20, t_pct=0.1)
                    sa = RHCRunner(
                        problem=problem,
                        experiment_name='dontcare',
                        output_directory='./results/grid_search_continuous_peaks_rhc',
                        seed=SEED,
                        max_attempts=max_attempts,
                        iteration_list=[iterations],
                        restart_list = [restarts],
                    )
                    _, df_run_curves = sa.run()
                    df_run_curves['group_number'] = group_i
                    df_run_curves['run_number'] = run_i
                    df_run_curves['problem_size'] = problem_size
                    df_run_curves['max_iterations'] = iterations
                    df_run_curves['max_attempts'] = max_attempts
                    df_run_curves['max_restarts'] = restarts
                    all_df = pd.concat([all_df, df_run_curves], axis=0)
                    run_i += 1
                group_i += 1
all_df.reset_index(inplace=True, drop=True)

In [ ]:
all_df.to_csv('temp.csv', index=False)

In [ ]:
# df = pd.read_csv('temp.csv')

In [ ]:
len(all_df['run_number'].unique())

In [ ]:
all_df['Fitness'].max()

In [ ]:
for run_i in sorted(all_df['run_number'].unique()):
    mask = all_df['run_number'] == run_i
    temp_df = all_df[mask]
    all_df.loc[mask, 'total_iterations'] = temp_df['Iteration'].max()
    all_df.loc[mask, 'total_time'] = temp_df['Time'].max()
    all_df.loc[mask, 'best_fitness'] = temp_df['Fitness'].max()
    all_df.loc[mask, 'total_fevals'] = temp_df['FEvals'].max()
    all_df.loc[mask, 'total_restarts'] = temp_df['current_restart'].max()
run_df = all_df[[
    'run_number', 'group_number', 'problem_size', 'max_iterations', 'max_attempts', 'max_restarts',
    'total_iterations', 'total_time', 'best_fitness', 'total_fevals', 'total_restarts']]
run_df.drop_duplicates(inplace=True)
len(run_df)

In [ ]:
for group_i in sorted(run_df['group_number'].unique()):
    mask = run_df['group_number'] == group_i
    temp_df = run_df[mask]
    for key in 'total_iterations', 'total_time', 'best_fitness', 'total_fevals', 'total_restarts':
        run_df.loc[mask, f"{key}_mean"] = temp_df[key].mean()
        run_df.loc[mask, f"{key}_std"] = temp_df[key].std()
group_df = run_df[[
    'group_number', 'problem_size', 'max_iterations', 'max_attempts', 'max_restarts',
    'total_iterations_mean', 'total_iterations_std', 'total_time_mean', 'total_time_std', 'best_fitness_mean',
    'best_fitness_std', 'total_fevals_mean', 'total_fevals_std', 'total_restarts_mean', 'total_restarts_std']]
group_df.drop_duplicates(inplace=True)
len(group_df)

In [ ]:
group_df

In [ ]:
max_fit = group_df['best_fitness_mean'].max()
best_df = group_df[group_df['best_fitness_mean'] == max_fit]
len(best_df)

In [ ]:
best_df['total_time_mean'].hist(bins=100)

In [ ]:
best_df['total_fevals_mean'].hist(bins=100)

In [ ]:
min_time = best_df['total_time_mean'].min()
min_fevals = best_df['total_fevals_mean'].min()
min_time, min_fevals

In [ ]:
best_df[best_df['total_time_mean'] == min_time]['total_fevals_mean']

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(best_df['total_time_mean'], best_df['total_fevals_mean'], 'o')

In [ ]:
best_df[best_df['total_time_mean'] == min_time]